# GraphRAG demo: Neo4j + LangChain + Gemini

**What we will do**
1. Write two small text files (our "documents")
2. Load them and split them into chunks
3. Ask Gemini to extract **entities** (nodes) and **relationships**
4. Store them in **Neo4j** (LangChain writes the Cypher `MERGE` for us)
5. Ask questions in plain English → Gemini writes a **Cypher** query → Neo4j answers

**Before you start**
- Neo4j running locally (Neo4j Desktop or Docker) at `bolt://localhost:7687`
- In Neo4j Desktop, install the **APOC** plugin for your database (LangChain uses it to read the graph schema)
- A Google AI Studio API key: https://aistudio.google.com/apikey

## Step 0 · Install packages

In [ ]:
%pip install -q langchain langchain-community langchain-neo4j langchain-experimental langchain-google-genai

## Step 1 · Settings

In [ ]:
import os
from getpass import getpass

# Google AI Studio key (typed in, not saved in the notebook)
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Google AI Studio API key: ")

# Your local Neo4j details
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "your_password_here"   # <-- change this

# Any free-tier Gemini model from AI Studio works here
GEMINI_MODEL = "gemini-3.6-flash"

## Step 2 · Create two small input files

In [ ]:
os.makedirs("data", exist_ok=True)

doc1 = """Alice is a data engineer at Neo4j. Alice lives in London.
Bob is a machine learning engineer at Google. Bob lives in Bengaluru.
Alice and Bob both studied at IIT Hyderabad."""

doc2 = """Neo4j has an office in London.
Google has offices in Bengaluru and Hyderabad.
Alice works on the GraphRAG project. Bob works on the Gemini project.
The GraphRAG project uses Gemini."""

with open("data/team.txt", "w") as f:
    f.write(doc1)
with open("data/projects.txt", "w") as f:
    f.write(doc2)

print("Created data/team.txt and data/projects.txt")

## Step 3 · Load the files and split them into chunks

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = TextLoader("data/team.txt").load() + TextLoader("data/projects.txt").load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents -> {len(chunks)} chunks")
for c in chunks:
    print("-", c.page_content[:80].replace("\n", " "), "...")

## Step 4 · Connect to Neo4j and the Gemini LLM

In [ ]:
from langchain_neo4j import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)

# Start with an empty database so the demo is repeatable
graph.query("MATCH (n) DETACH DELETE n")
print("Connected to Neo4j and cleared old data")

## Step 5 · LLM extracts entities and relationships

`LLMGraphTransformer` sends each chunk to Gemini. Gemini returns structured data
(nodes + relationships), and LangChain turns it into **GraphDocument** objects.
No Cypher yet. We also tell it which node and relationship types are allowed, so the graph stays clean.

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Company", "City", "University", "Project"],
    allowed_relationships=["WORKS_AT", "LIVES_IN", "STUDIED_AT",
                           "HAS_OFFICE_IN", "WORKS_ON", "USES"],
)

graph_docs = transformer.convert_to_graph_documents(chunks)

# Look at what the LLM extracted
for gd in graph_docs:
    print("NODES:")
    for n in gd.nodes:
        print(f"   {n.id} ({n.type})")
    print("RELATIONSHIPS:")
    for r in gd.relationships:
        print(f"   {r.source.id} -[{r.type}]-> {r.target.id}")
    print()

## Step 6 · Store the graph in Neo4j

`add_graph_documents` turns the GraphDocuments into Cypher `MERGE` statements and runs them.
`include_source=True` also saves each chunk as a `Document` node linked with `MENTIONS`.

In [ ]:
graph.add_graph_documents(graph_docs, include_source=True, baseEntityLabel=True)

graph.refresh_schema()
print(graph.schema)

### See the graph
Open Neo4j Browser at http://localhost:7474 and run:
```cypher
MATCH (n) RETURN n
```
Or check it from Python with a Cypher query we write ourselves:

In [ ]:
rows = graph.query("""
MATCH (a)-[r]->(b)
WHERE NOT a:Document
RETURN a.id AS from, type(r) AS relationship, b.id AS to
""")
for row in rows:
    print(f"{row['from']} -[{row['relationship']}]-> {row['to']}")

## Step 7 · Ask questions

`GraphCypherQAChain` does this for every question:
1. Gemini reads the graph schema + your question and **writes a Cypher query**
2. Neo4j runs the query and returns rows
3. Gemini turns the rows into a plain-English answer

`verbose=True` prints the generated Cypher so students can see it.

In [ ]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,                    # show the generated Cypher
    validate_cypher=True,            # fix relationship directions if needed
    allow_dangerous_requests=True,   # required: the LLM runs queries on your DB
)

In [ ]:
result = chain.invoke({"query": "Where does Alice work?"})
print("ANSWER:", result["result"])

In [ ]:
questions = [
    "Who studied at IIT Hyderabad?",
    "In which cities does Google have offices?",
    "Which project does Bob work on?",
    "Which company has an office in the city where Alice lives?",   # multi-hop
]

for q in questions:
    print("\nQUESTION:", q)
    print("ANSWER:", chain.invoke({"query": q})["result"])

## Try it yourself
- Add a third text file and re-run Steps 2–6
- Ask your own question in the cell above
- Compare: a question that needs **two hops** (like the last one) is where GraphRAG beats plain vector RAG

**If something fails**
- *APOC error* → install the APOC plugin in Neo4j Desktop and restart the database
- *429 / quota error* → free-tier rate limit; wait a minute and re-run the cell
- *Empty answer* → print the Cypher (`verbose=True`) and check the node names match what Step 5 extracted